In [1]:
# Structure: Train MLPClassifier (non deterministic) with the load_breast_cancer dataset
# Explainability: Global LIME values with LimeTabularExplainer & SubmodularPick
# Script: Run 10 experiments, keep all feature importance values and measure 
# and measure the Mean of the ranking deviation + average std. deviation of the explanations
# Runtime: 3 hours for 125 samples / 7 min for 5 samples

from lightgbm import LGBMRegressor
from sklearn.datasets import fetch_california_housing
import numpy as np
import lime
import pandas as pd
import numpy as np
import lime.lime_tabular
import warnings
from sklearn.model_selection import train_test_split
import sys
import os
import statistics

In [2]:
X, y = fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = LGBMRegressor()
model.fit(X_train, y_train)

feature_names = fetch_california_housing().feature_names
categorical_features = np.argwhere(np.array([len(set(X_train[:,x])) for x in range(X_train.shape[1])]) <=10).flatten()

explainer = lime.lime_tabular.LimeTabularExplainer(X_train, feature_names=feature_names, 
                                                categorical_features = categorical_features, verbose=True, mode='regression')

# Choose which instance to examine
input = 24

In [3]:
# Grab global explanations
sample_size = 5
from lime import submodular_pick
sp_obj = submodular_pick.SubmodularPick(explainer, X, model.predict, sample_size=sample_size, num_features=10, num_exps_desired=5)


Intercept 2.238463866638004
Prediction_local [2.71306622]
Right: 3.6154538986615377
Intercept 2.663105782008824
Prediction_local [1.42226753]
Right: 0.972569745748422
Intercept 2.842990560919029
Prediction_local [0.88473353]
Right: 0.8865853578444935
Intercept 1.771485278754589
Prediction_local [3.98735837]
Right: 4.708547717750724
Intercept 2.001024989527912
Prediction_local [3.42170455]
Right: 3.1248394988408656


In [6]:
W_pick=pd.DataFrame([dict(this.as_list()) for this in sp_obj.explanations])

df = W_pick.mean(axis=0)
df = abs(df.head())

values = df.values.tolist()
features = df.index.to_list()

sum_list = []
for i in range(len(features)):
    sum_list.append(features[i]+", "+str(values[i]))

feature_names = load_diabetes().feature_names

In [7]:
# Extract LIME explanations
feature_importance = []
for i in range(len(sum_list)):
    for word in feature_names:
    # if a word from the features list is present in the explanation strings (1st element of the list) then grab it
        if word in sum_list[i]:
            # transfer the feature and its value to a 
            feature_importance.append("%s = " %word + str("{:.3f}".format(float(sum_list[i].split(",")[1]))))

# Save explanations of each run to a temp file         
with open(r"C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_LIME\features.txt", "a") as file:
        file.write("Run number: %d\n\n" %211)
        for row in feature_importance:
            s = "".join(map(str, row))
            file.write(s+'\n')
        file.write('\n')
file.close()

In [10]:
with open(r'C:\Users\Stefanos Rigas\Documents\Projects\Thesis\thesis\robustness-evaluation\explanations\G_E_LIME\features.txt','r') as file:
    data = file.readlines()

file.close()

#-----------------------------------------------------------------------------------------------------------------
    
# Make script that compares the  data - explanations, and the highlighted features (if top feature is the same)

# This is a function for getting the change % of a feature (will not be used here)
def get_change(current, previous):
    
    if current == previous:
        return 100.0
    try:
        change = (abs(current - previous) / ((current + previous)/2) ) * 100.0
        return "{:.2f}".format(change)
    except ZeroDivisionError:
        return 0

values = []
features = []
for i in range(10):
    for j in range(5):
        x = data[i*8+j+2].split()
        values.append(x[2])
        features.append(x[0])